# Run the experiment on Google Colab

Open this notebook from its GitHub link in every session, on a GPU runtime (Runtime > Change
runtime type), so the cells are always those on `main`:
<https://colab.research.google.com/github/meiazero/garimpo-uncertainty/blob/main/notebooks/01_run_experiment.ipynb>

1. Set the form in section 3: the scale, the models, and a stage to redo after a code fix.
2. Run Runtime > Run all and keep the tab open.
3. Read the verdicts at the end of section 5.

Every cell can run again: Run all a second time, in the same session or a new one, skips what is
done and resumes the rest. The last cell saves everything to Drive and releases the GPU, so the
session stops spending compute units. The data must already be in Drive as
`MyDrive/garimpo/data.tar.zst`; the notebook downloads nothing and analyzes nothing.

| Section | What it does |
|---|---|
| 1. Setup | Mounts Drive, clones the code, builds the environment |
| 2. Data | Unpacks the data once, restores earlier runs and logs from Drive |
| 3. What to run | The form, and the values each run will use |
| 4. Tests | The test suite on the GPU, when the form asks for it |
| 5. Runs | Each chosen model through every stage, then one verdict per model |
| 6. Merge | The four runs' tables, once all four passed |
| 7. Release | The last copy to Drive, then the GPU released |

## 1. Setup

The runtime's disk is erased when a session ends, so this cell clones the code and builds the
environment again. It clones only when the clone is missing, so it is safe to run again. The
install stops, and names the package, when pip would replace Colab's torch or a CUDA library,
which pick the GPU's kernels: choose the latest runtime version then.
`DATA` must stay `/content/garimpo-data`: runs record the path of their data and refuse another.

In [ ]:
# @title Setup
import os
import subprocess
import threading
import time
from pathlib import Path

from google.colab import drive, runtime

REPO = "https://github.com/meiazero/garimpo-uncertainty.git"
CODE = "/content/garimpo-uncertainty"  # the clone; erased with the session
DRIVE = "/content/drive/MyDrive/garimpo"  # kept between sessions
DATA = "/content/garimpo-data"  # runs record this path, so it never changes
os.environ["GARIMPO_DATA_DIR"] = DATA
SESSION_START = globals().get("SESSION_START", time.time())  # older logs hold no attempt of ours
periodic_copy = globals().get("periodic_copy")  # the copy to Drive every 30 minutes, once started


def copy_to_drive_every_30_min(stop: threading.Event) -> None:
    """Copy runs/ and the logs to Drive every 30 minutes until stop is set."""
    runs = ["rsync", "-a", "--exclude", "*.tmp", "runs/", f"{DRIVE}/runs/"]
    logs = ["rsync", "-a", "outputs/logs/", f"{DRIVE}/logs/"]
    while not stop.wait(1800):
        for command in (runs, logs):
            try:
                copied = subprocess.run(command, check=False)  # noqa: S603
            except OSError as error:
                print(f"The copy to Drive every 30 minutes failed: {error}")
                continue
            if copied.returncode != 0:
                print(f"The copy to Drive every 30 minutes ended with code {copied.returncode}")


def release_runtime() -> None:
    """Write pending files to Drive, then release the runtime, which ends the session."""
    try:
        drive.flush_and_unmount()
    except Exception as error:  # noqa: BLE001 -- the runtime must be released whatever failed
        print(f"Drive did not confirm the last writes; check the files in Drive: {error}")
    runtime.unassign()


drive.mount("/content/drive")
!apt-get -qq install -y rsync zstd > /dev/null
if not Path(CODE, ".git").is_dir():
    !git clone --recurse-submodules {REPO} {CODE}
%cd {CODE}
!python scripts/colab_install.py
!mkdir -p {DRIVE}/runs {DRIVE}/logs {DATA} runs outputs/logs
!git log -1 --format='Running commit %h %cs %s'

## 2. Data

The store is unpacked from Drive only when `DATA` does not hold a complete unpack yet, marked by
`.unpacked` once tar succeeds: it takes 69.2 GB, for all
18,864 cells, whatever the training fraction. The runs and logs of earlier sessions come back from
Drive with `--ignore-existing`, so finished stages are skipped and the local copies, never older
than Drive's, are kept. If the restore prints an error, interrupt Run all and fix it first: a
training run without its earlier checkpoint starts over and overwrites the one in Drive.

In [ ]:
# @title Unpack the data and restore earlier runs
# The marker is written only after tar succeeds, so a cut unpack is redone, never trusted.
if Path(DATA, ".unpacked").exists():
    print(f"The data is already in {DATA}.")
else:
    !tar -I zstd -xf {DRIVE}/data.tar.zst -C {DATA}
    if globals().get("_exit_code") == 0:
        Path(DATA, ".unpacked").touch()
!rsync -a --ignore-existing --exclude '*.tmp' {DRIVE}/runs/ runs/
!rsync -a --ignore-existing {DRIVE}/logs/ outputs/logs/
!df -h /content

## 3. What to run

- `SCALE`: `pilot` trains on 1% of the training tiles for a few updates, to check the code;
  `full` is the experiment. `auto` continues the full runs once one has a checkpoint in Drive,
  and runs the pilot otherwise.
- The model boxes: the models this Run all takes through the stages, in this order.
- `FORCE_FROM`: after a code fix, the first stage the fix touches. That stage and the ones after it
  run again for every chosen model; earlier ones are skipped. Leave it empty to run only what is
  not finished. It refuses `train` while `last.pt` holds a training that has not finished.
- `FORCE_VALIDATE`: check the data again.
- `RUN_TESTS`: run the test suite on the GPU first, about the first session after a code change.

The form's values stay set in the open tab only; reopening the notebook from its link restores
these defaults. Every other value comes from `configs/`, versioned with the code: change it on your
own machine, push, and the next session clones it.

In [ ]:
# @title The form
SCALE = "auto"  # @param ["auto", "pilot", "full"]
UNET_SCRATCH = True  # @param {type:"boolean"}
UNET_IMAGENET = True  # @param {type:"boolean"}
UNET_SENTINEL2 = True  # @param {type:"boolean"}
CONVNET = True  # @param {type:"boolean"}
FORCE_FROM = ""  # @param ["", "train", "infer", "fit_directions", "layer_monotonicity", "choose_layer", "endpoint_monotonicity", "compass_scores", "calibrate", "generate_intervals", "compute_metrics", "search_profile"]
FORCE_VALIDATE = False  # @param {type:"boolean"}
RUN_TESTS = False  # @param {type:"boolean"}

MODELS = [
    model
    for model, chosen in (
        ("unet_scratch", UNET_SCRATCH),
        ("unet_imagenet", UNET_IMAGENET),
        ("unet_sentinel2", UNET_SENTINEL2),
        ("convnet", CONVNET),
    )
    if chosen
]
OPTIONS = (f"--force-from {FORCE_FROM}" if FORCE_FROM else "") + (
    " --force-validate" if FORCE_VALIDATE else ""
)
try:
    chosen_scale = subprocess.run(  # noqa: S603
        ["garimpo", "status", SCALE, "--choose"],  # noqa: S607
        capture_output=True,
        text=True,
        check=False,
    )
    if chosen_scale.returncode == 0:
        SCALE = chosen_scale.stdout.splitlines()[-1]
    else:
        print(chosen_scale.stdout, chosen_scale.stderr)
except OSError as error:
    # A raise here would cancel the release queued at the end; left at auto, every run refuses.
    print(f"Could not choose the scale: {error!r}")
print(f"Scale {SCALE}; models {', '.join(MODELS) or 'none'}; options {OPTIONS or 'none'}.")
!garimpo status {SCALE}

## 4. Tests

Runs only with `RUN_TESTS` checked. The tests build a small dataset of their own and run on CUDA
here, where none is skipped. A failure does not stop the cells after it: read this output before
you trust the runs.

In [ ]:
# @title Run the tests on the GPU
if RUN_TESTS:
    !GARIMPO_TEST_DEVICE=cuda pytest -q
else:
    print("Skipped: RUN_TESTS is not checked.")

## 5. Runs

Each chosen model runs through every stage with `garimpo run`, one after another on the GPU. Each
prints one line per stage with its time and key numbers, writes everything to
`outputs/logs/<scale>-<model>.log`, and ends with `PIPELINE CHECK: PASS` or `FAIL`. A model that
fails does not stop the next one. A copy of `runs/` and the logs to Drive every 30 minutes keeps the
finished stages if the session ends early; Run all again in a new session resumes from there.

The verdicts, one line per model: `PASS` means every stage finished and every check held. After
`FAIL`, the log names the stage that failed. `NO VERDICT`: the job died before its check.
`NO LOG`: no log yet. `NOT RUN`: this session wrote nothing to that model's log.

Interrupting this cell once stops the current stage, still prints its check, and runs no later
model; then run section 7 by hand to save and release.

In [ ]:
# @title Run the chosen models
if periodic_copy is None or not periodic_copy.is_alive():
    stop_copy = threading.Event()
    periodic_copy = threading.Thread(
        target=copy_to_drive_every_30_min, args=(stop_copy,), daemon=True
    )
    periodic_copy.start()
INTERRUPTED = 130
for model in MODELS:
    print(f"\n===== {model} =====")
    !garimpo run {SCALE} {model} {OPTIONS}
    # garimpo run answers an interrupt with 130; killed by a signal, its negative.
    if globals().get("_exit_code", 0) in {INTERRUPTED, -2, -9, -15}:
        print("Interrupted: the models after this one did not run.")
        break
!garimpo status {SCALE} --since {SESSION_START}

## 6. Merge

Merges the four runs' tables: into `runs/pilot/tables/` for the pilot, into `outputs/` for the full
runs, then copies `outputs/` to Drive. It refuses, and writes nothing, unless every stage of all
four runs is `ok`, the other layers included. After the full runs, download
`MyDrive/garimpo/outputs/` and commit it from your own machine: these are the final tables.

In [ ]:
# @title Merge the four runs
TABLES = "runs/pilot/tables" if SCALE == "pilot" else "outputs"
!garimpo merge --out {TABLES} --scale {SCALE}
!rsync -a outputs/ {DRIVE}/outputs/

## 7. Release

Stops the copy every 30 minutes, after any copy in progress, copies the runs and logs to Drive one
last time, and releases the GPU. No cell before this one raises a Python error, so Run all always
reaches it.

In [ ]:
# @title Save to Drive and release the GPU
if periodic_copy is not None:
    stop_copy.set()
    periodic_copy.join()
!rsync -a --exclude '*.tmp' runs/ {DRIVE}/runs/
!rsync -a outputs/logs/ {DRIVE}/logs/
release_runtime()

## Where the files are

| Path | Contents | Kept after the session |
|---|---|---|
| `MyDrive/garimpo/data.tar.zst` | The data, which the notebook only reads | Yes |
| `MyDrive/garimpo/runs/` | Every run: checkpoints, tables, stage statuses | Yes |
| `MyDrive/garimpo/logs/` | The log of every pilot and full-run job | Yes |
| `MyDrive/garimpo/outputs/` | The merged tables of the full runs | Yes |
| `/content/garimpo-uncertainty/` | The code | No |
| `/content/garimpo-data/` | The unpacked data | No |